# Conformal geometry: circles, spheres, and intersections

**CGA2** represents points, lines, circles, and their point-pair intersections.
**CGA3** adds spheres and circle intersections with planes. Build the objects
with the exterior product `|`, intersect them with the regressive product
`&`, and interpolate their motions with algebra elementary functions.

The sliders run on Plotly frames; drag the 3D figure to inspect a different
view. Run this notebook in order, using the examples environment described
in [`README.md`](README.md).

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.offline import init_notebook_mode
from algpy.algebras import cga2, cga3
from _geometry import (cga_frame, cga_point, cga_plane, cga_coordinates,
                       cga_sphere_data, cga_point_pair, cga_circle_data)
from _plotting import slider_controls

init_notebook_mode(connected=False)
pio.templates.default = "plotly_white"

## The two special generators come first

`cga2.gens()` returns $(e_-,e_+,e_x,e_y)$; `cga3` adds $e_z$.
The first two squares are $-1,+1$. Define
\[
o=(e_- - e_+)/2,\quad n_\infty=e_-+e_+,\quad
X(x)=o+x+\tfrac12|x|^2 n_\infty.
\]
Then $X(x)^2=0$, $X(x)\cdot n_\infty=-1$, and
$X(x)\cdot X(y)=-|x-y|^2/2$. For vectors, the inner product is the scalar part
of the geometric product, `(a*b)[0]`. A nonzero homogeneous rescaling represents
the same point. These conventions match the library's generator order.

In [ ]:
origin2, infinity2, axes2 = cga_frame(cga2)
P = cga_point(cga2, [.4, -.2])
Q = cga_point(cga2, [-.3, .5])
np.testing.assert_allclose((P*P)[0], 0, atol=1e-14)
np.testing.assert_allclose((P*infinity2)[0], -1, atol=1e-14)
np.testing.assert_allclose((P*Q)[0], -.5*np.sum((np.array([.4, -.2])-[-.3, .5])**2))
np.testing.assert_allclose(cga_coordinates(3*P), [.4, -.2])
print("Decoded null point:", cga_coordinates(P))

## CGA2: join three points, meet a line

Use the **outer-product representation**:
$C=X_1\wedge X_2\wedge X_3$ is a circle and
$L=X_a\wedge X_b\wedge n_\infty$ is a line. The CGA types write the
exterior product as `|` and the regressive product as `&`, their lattice
join and meet. The regressive product intersects the objects; it agrees
with the metric-dual formula $C\cap L=\big(C^*\wedge L^*\big)I$, where
$A^*=AI^{-1}$, up to a sign, which does not change the represented object.
The meet is a grade-two **point pair**, rather than one point. `_geometry.py`
factors this bivector into its two null points using geometric products.
We choose distinct real intersections throughout the demonstration.

A dual circle/sphere vector has the form
$S=X(c)-\tfrac12r^2n_\infty$, from which its center and radius can be read
after normalizing $S\cdot n_\infty=-1$.

In [ ]:
angles2 = np.arange(3)*2*np.pi/3
anchors2 = [cga_point(cga2, [np.cos(t), np.sin(t)]) for t in angles2]
circle2 = anchors2[0] | anchors2[1] | anchors2[2]
line_height = .15
line2 = (cga_point(cga2, [-2, line_height])
         | cga_point(cga2, [2, line_height]) | infinity2)
pair2 = circle2 & line2
intersections2 = cga_point_pair(pair2)
center2, radius2 = cga_sphere_data(circle2)
np.testing.assert_allclose(center2, [0, 0], atol=1e-13)
np.testing.assert_allclose(radius2, 1, atol=1e-13)
for point in intersections2:
    np.testing.assert_allclose((point*point)[0], 0, atol=1e-12)
    np.testing.assert_allclose(cga2.to_float(circle2 | point), 0, atol=1e-12)
    np.testing.assert_allclose(cga2.to_float(line2 | point), 0, atol=1e-12)
    np.testing.assert_allclose(cga_coordinates(point)[1], line_height, atol=1e-12)
print("Line-circle meet:", np.array([cga_coordinates(point) for point in intersections2]))

## Interpolating a conformal motion

CGA also handles uniform scale. With $D=e_-e_+$,
$\exp(\log(s)D/2)$ scales Euclidean coordinates by $s$.
Compose translation, rotation, and dilation, then interpolate from the
identity with `one.mix(target, t)`. Sandwiching and renormalizing the null
vectors gives the Euclidean positions.

In [ ]:
em2, ep2, ex2, ey2 = cga2.gens()
target2 = (np.exp(-.5*(.6*ex2 + .2*ey2)*infinity2)
           * np.exp(-.35*ex2*ey2)
           * np.exp(np.log(1.4)/2*em2*ep2))
sample_point = cga_point(cga2, [1, 0])
moved = (target2*sample_point*target2.inv()).grade[1]
np.testing.assert_allclose(cga_coordinates(moved),
                           1.4*np.array([np.cos(.7), np.sin(.7)]) + [.6, .2], atol=2e-11)

In [ ]:
circle_angles = np.linspace(0, 2*np.pi, 100)

def conformal_scene2(t):
    motor = cga2.one().mix(target2, float(t))
    anchors = [(motor*point*motor.inv()).grade[1] for point in anchors2]
    circle = anchors[0] | anchors[1] | anchors[2]
    center, radius = cga_sphere_data(circle)
    pair = circle & line2
    points = cga_point_pair(pair)
    coordinates = np.array([cga_coordinates(point) for point in points])
    coordinates = coordinates[np.argsort(coordinates[:, 0])]
    np.testing.assert_allclose(coordinates[:, 1], line_height, atol=2e-10)
    np.testing.assert_allclose(np.linalg.norm(coordinates-center, axis=1), radius, atol=2e-10)
    curve = center + radius*np.column_stack([np.cos(circle_angles), np.sin(circle_angles)])
    anchor_coordinates = np.array([cga_coordinates(point) for point in anchors])
    return [
        go.Scatter(x=curve[:, 0], y=curve[:, 1], mode="lines", line_color="#2563eb", name="Joined circle"),
        go.Scatter(x=anchor_coordinates[:, 0], y=anchor_coordinates[:, 1], mode="markers",
                   marker=dict(color="#2563eb", size=7), name="Three defining points"),
        go.Scatter(x=coordinates[:, 0], y=coordinates[:, 1], mode="markers",
                   marker=dict(color="#db2777", size=12), name="Decoded meet point pair"),
    ]

times = np.linspace(0, 1, 31)
fig = go.Figure(data=[go.Scatter(x=[-2, 2.6], y=[line_height]*2, mode="lines",
                                 line=dict(color="#64748b", dash="dash"), name="Fixed line")]
                          + conformal_scene2(0))
fig.frames = [go.Frame(name=str(index), data=conformal_scene2(t), traces=[1, 2, 3])
              for index, t in enumerate(times)]
fig.update_layout(title="CGA2: interpolate a circle and recompute its line intersection", height=580)
fig.update_xaxes(title="x", range=[-2, 2.6])
fig.update_yaxes(title="y", range=[-1.8, 2.1], scaleanchor="x", scaleratio=1)
slider_controls(fig, [f"{t:.2f}" for t in times]).show(auto_play=False)

## CGA3: a sphere meets a plane in a circle

Four noncoplanar null points join to a sphere, a grade-four blade.
The vector $\pi=e_z+h n_\infty$ represents the plane $z=h$ in the
inner-product model; $\pi I$ is its outer-product representation.

Their meet `sphere & plane` is a grade-three circle. The helper `cga_circle_data` recovers
its plane and centered sphere using contractions of its metric dual.
All sampled circle points are checked against the original sphere and plane.

In [ ]:
origin3, infinity3, axes3 = cga_frame(cga3)
em3, ep3, ex3, ey3, ez3 = cga3.gens()
directions3 = np.array([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]])/np.sqrt(3)
initial_center3 = np.array([0., 0., .15])
initial_radius3 = 1.2
anchors3 = [cga_point(cga3, initial_center3 + initial_radius3*direction) for direction in directions3]
sphere3 = anchors3[0] | anchors3[1] | anchors3[2] | anchors3[3]
plane_height = .2
plane3 = cga_plane(cga3, [0, 0, 1], plane_height)  # z = plane_height
circle3 = sphere3 & plane3
center3, radius3, normal3 = cga_circle_data(circle3)
np.testing.assert_allclose(center3, [0, 0, plane_height], atol=2e-12)
np.testing.assert_allclose(radius3, np.sqrt(initial_radius3**2-(plane_height-.15)**2), atol=2e-12)
print("Circle center:", center3, "radius:", radius3, "plane normal:", normal3)

In [ ]:
target3 = (np.exp(-.5*(.45*ex3 - .25*ey3 + .35*ez3)*infinity3)
           * np.exp(-.325*ex3*ez3)
           * np.exp(np.log(1.2)/2*em3*ep3))
sphere_u, sphere_v = np.meshgrid(np.linspace(0, 2*np.pi, 25), np.linspace(0, np.pi, 17))
sphere_directions = np.stack([np.sin(sphere_v)*np.cos(sphere_u),
                               np.sin(sphere_v)*np.sin(sphere_u), np.cos(sphere_v)], axis=-1)

def conformal_scene3(t):
    motor = cga3.one().mix(target3, float(t))
    anchors = [(motor*point*motor.inv()).grade[1] for point in anchors3]
    sphere = anchors[0] | anchors[1] | anchors[2] | anchors[3]
    center, radius = cga_sphere_data(sphere)
    circle = sphere & plane3
    circle_center, circle_radius, normal = cga_circle_data(circle)
    expected_radius = np.sqrt(radius**2 - (center[2]-plane_height)**2)
    np.testing.assert_allclose(circle_center, [center[0], center[1], plane_height], atol=3e-10)
    np.testing.assert_allclose(circle_radius, expected_radius, atol=3e-10)
    tangent = np.cross(normal, [1., 0, 0])
    tangent /= np.linalg.norm(tangent)
    other = np.cross(normal, tangent)
    circle_points = circle_center + circle_radius*(np.cos(circle_angles)[:, None]*tangent
                                                   + np.sin(circle_angles)[:, None]*other)
    for row in circle_points[::33]:
        point = cga_point(cga3, row)
        np.testing.assert_allclose(cga3.to_float(sphere | point), 0, atol=2e-9)
        np.testing.assert_allclose(cga3.to_float(plane3 | point), 0, atol=2e-9)
    mesh = center + radius*sphere_directions
    anchor_coordinates = np.array([cga_coordinates(point) for point in anchors])
    return [
        go.Surface(x=mesh[..., 0], y=mesh[..., 1], z=mesh[..., 2], opacity=.23,
                   colorscale=[[0, "#60a5fa"], [1, "#60a5fa"]], showscale=False, name="Sphere"),
        go.Scatter3d(x=circle_points[:, 0], y=circle_points[:, 1], z=circle_points[:, 2],
                     mode="lines", line=dict(color="#db2777", width=7), name="Meet circle"),
        go.Scatter3d(x=anchor_coordinates[:, 0], y=anchor_coordinates[:, 1], z=anchor_coordinates[:, 2],
                     mode="markers", marker=dict(size=4, color="#2563eb"), name="Four defining points"),
        go.Scatter3d(x=[center[0], circle_center[0]], y=[center[1], circle_center[1]],
                     z=[center[2], circle_center[2]], mode="markers",
                     marker=dict(size=5, color=["#2563eb", "#db2777"]), name="Sphere / circle centers"),
    ]

plane_x, plane_y = np.meshgrid([-2., 2.4], [-2., 2.])
fig = go.Figure(data=[go.Surface(x=plane_x, y=plane_y, z=np.full_like(plane_x, plane_height),
                                 opacity=.15, showscale=False,
                                 colorscale=[[0, "#94a3b8"], [1, "#94a3b8"]], name="Fixed plane")]
                          + conformal_scene3(0))
fig.frames = [go.Frame(name=str(index), data=conformal_scene3(t), traces=[1, 2, 3, 4])
              for index, t in enumerate(times)]
fig.update_layout(title="CGA3: interpolate a sphere and compute its plane-section circle", height=680,
                  scene=dict(xaxis=dict(range=[-2.5, 2.5], title="x"),
                             yaxis=dict(range=[-2.5, 2.5], title="y"),
                             zaxis=dict(range=[-2.5, 2.5], title="z"), aspectmode="cube"))
slider_controls(fig, [f"{t:.2f}" for t in times]).show(auto_play=False)

**Try changing** the target translation, scale, or plane height. Keep two
distinct real intersections in CGA2 and a real, nonzero-radius section circle
in CGA3. Tangent, imaginary, and ideal objects need additional decoding cases;
the algebraic meet still describes those configurations.

The next notebooks use commuting dual generators to build Jacobians,
Hessians, normals, and curvature from NumPy expressions.